In [ ]:
"""Day 26 / Session 1 - Lab Assignment 2: Lexicon Sentiment Scoring & Few-Shot Prompt Formatting."""
import re

POS = {"good", "great", "love", "excellent", "happy"}
NEG = {"bad", "terrible", "hate", "awful", "sad"}
NEGATORS = {"not", "no", "never"}
texts = ["i love it", "so happy", "just awful", "very sad", "the movie exists"]
labels = ["positive", "positive", "negative", "negative", "neutral"]


def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def sentiment_score(text, pos_words, neg_words):
    tokens = tokenize(text)
    total = 0
    for i, tok in enumerate(tokens):
        value = 1 if tok in pos_words else -1 if tok in neg_words else 0
        if value and i > 0 and tokens[i - 1] in NEGATORS:        # an immediately preceding negator flips the value
            value = -value
        total += value
    return int(total)


def classify_sentiment(text, pos_words, neg_words):
    score = sentiment_score(text, pos_words, neg_words)
    return "positive" if score > 0 else "negative" if score < 0 else "neutral"


def evaluate(texts, labels, pos_words, neg_words):
    if len(texts) == 0:
        return 0.0
    right = sum(classify_sentiment(t, pos_words, neg_words) == y for t, y in zip(texts, labels))
    return float(right / len(texts))


def build_few_shot_prompt(examples, query, instruction="Classify the sentiment of the review as positive or negative."):
    blocks = [f"Review: {text}\nSentiment: {label}" for text, label in examples]
    blocks.append(f"Review: {query}\nSentiment:")                  # dangling label: the model completes it
    return instruction + "\n\n" + "\n\n".join(blocks)
